# Scriptorium — Smoke Test

Tiny (~100 samples, 2 epochs) run to verify the save/copy path works end-to-end. Takes ~5–10 min on T4.

Same setup as `train_kaggle.ipynb`: **Settings → Accelerator → GPU T4 x2**, then **Save Version → Save & Run All (Commit)**.

In [ ]:
!pip install -q kraken
!kraken --version

In [ ]:
%cd /kaggle/working
!rm -rf scriptorium-rails
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/kraftinator/scriptorium-rails.git
%cd scriptorium-rails
!git lfs pull -I data/barton_htr_data.tar.gz
!ls -lh data/barton_htr_data.tar.gz

In [ ]:
%cd /kaggle/working/scriptorium-rails
!mkdir -p data-extract
!tar xzf data/barton_htr_data.tar.gz -C data-extract
!sed -i 's|/Users/admin/scriptorium/data|/kaggle/working/scriptorium-rails/data-extract|g' data-extract/manifest.jsonl
!python python/prep_kraken_gt.py data-extract/manifest.jsonl
!find data-extract/crops -name 'line_*.png' | while read p; do [ -f "${p%.png}.gt.txt" ] && echo "$p"; done > data-extract/all_images.lst
!head -100 data-extract/all_images.lst > data-extract/smoke_images.lst
!wc -l data-extract/smoke_images.lst

In [ ]:
# Quick train: 2 epochs (fixed, no early-stopping) on 100 samples.
!mkdir -p /kaggle/working/models
!ketos -v -d cuda:0 --workers 2 train \
    -B 16 \
    -o /kaggle/working/models/smoke \
    -q fixed \
    -N 2 \
    -F 1.0 \
    -f path \
    -t data-extract/smoke_images.lst

In [ ]:
# Verify: what did kraken actually write?
!echo '--- full tree ---'
!find /kaggle/working/models -type f 2>/dev/null
!echo '--- copy to /kaggle/working root ---'
!best=$(ls -t /kaggle/working/models/smoke/best_*.safetensors 2>/dev/null | head -1) && \
  echo "safetensors: $best" && cp "$best" /kaggle/working/smoke.safetensors && \
  ls -lh /kaggle/working/smoke.safetensors || echo 'no safetensors'
!latest=$(ls -t /kaggle/working/models/smoke/*.ckpt 2>/dev/null | head -1) && \
  echo "ckpt: $latest" && cp "$latest" /kaggle/working/smoke.ckpt && \
  ls -lh /kaggle/working/smoke.ckpt || echo 'no ckpt'